# Exploration du dataset musical

Notebook d'exploration : chargement du dataset fusionné, vérification des features audio et de leur normalisation (StandardScaler) avant le KNN.

Le nettoyage et le dédoublonnage sont faits au lancement du backend (`data.py`), pas ici.

## 1. Chargement

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

DATASET_PATH = '../backend/dataset_merged_raw.csv'  

df = pd.read_csv(DATASET_PATH)
print(df.shape)
df.dtypes

(114000, 21)


Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object

## 2. Features audio

In [2]:
feature_cols = [
    'popularity', 'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
]
metadata_cols = ['track_id', 'artists', 'album_name', 'track_name']

df[feature_cols].describe()

,popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
count,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000
mean,33.238535,0.566800,0.641383,-8.258960,0.084652,0.314910,0.156050,0.213553,0.474068,122.147837
std,22.305078,0.173542,0.251529,5.029337,0.105732,0.332523,0.309555,0.190378,0.259261,29.978197
min,0.000000,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17.000000,0.456000,0.472000,-10.013000,0.035900,0.016900,0.000000,0.098000,0.260000,99.218750
50%,35.000000,0.580000,0.685000,-7.004000,0.048900,0.169000,0.000042,0.132000,0.464000,122.017000
75%,50.000000,0.695000,0.854000,-5.003000,0.084500,0.598000,0.049000,0.273000,0.683000,140.071000
max,100.000000,0.985000,1.000000,4.532000,0.965000,0.996000,1.000000,1.000000,0.995000,243.372000


## 3. Normalisation

Les features ont des échelles très différentes (`tempo` en BPM, `loudness` en dB, le reste entre 0 et 1). Sans normalisation, `tempo` et `loudness` domineraient la distance du KNN.

In [3]:
scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols])
print(X.shape)

pd.DataFrame(X, columns=feature_cols).describe().loc[['mean', 'std']].round(2)

(114000, 10)


,popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
mean,-0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 4. Vérifications ponctuelles

Exemple : retrouver les morceaux d'un artiste sans modifier `df` (le sous-ensemble est stocké dans une variable séparée).

In [4]:
drake = df.loc[df['artists'] == 'Drake', metadata_cols + ['popularity']]
drake.sort_values('popularity', ascending=False).head(10)

,track_id,artists,album_name,track_name,popularity
51250,6DCZcSspjsKoFjzjrWoCdn,Drake,Scorpion,God's Plan,84
